This code check the "androidmanifest.xml" for existance and at least one activity without cloning the repo

In [1]:
import os
import re
import pandas as pd
import base64
import requests
import xml.etree.ElementTree as ET
from dotenv import load_dotenv
from time import sleep

# Load all tokens from .env
load_dotenv("All_Tokens.env")
tokens = [v for k, v in os.environ.items() if k.startswith("GITHUB_TOKEN_") and v]
if not tokens:
    raise ValueError("No GitHub tokens found in All_Tokens.env")

token_index = 0
def get_headers():
    return {
        "Authorization": f"token {tokens[token_index]}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-manifest-checker"
    }

def rotate_token():
    global token_index
    token_index = (token_index + 1) % len(tokens)
    print(f"🔁 Rotated to token #{token_index + 1}")

# Input and output file paths
input_csv = r"D:\Android_Mobile_App\AndroidProject_dataset\Repo_List.csv"
output_csv = r"D:\Android_Mobile_App\AndroidProject_dataset\Repo_List_checked.csv"

# Load CSV
df = pd.read_csv(input_csv)
df["has_manifest"] = "no"
df["has_activity"] = "no"

# Loop through each repo
for idx, row in df.iterrows():
    full_name = row.get("full_name")
    if not isinstance(full_name, str) or "/" not in full_name:
        continue

    # Try standard path to AndroidManifest.xml
    path = "app/src/main/AndroidManifest.xml"
    url = f"https://api.github.com/repos/{full_name}/contents/{path}"

    success = False
    for _ in range(len(tokens)):
        response = requests.get(url, headers=get_headers())
        if response.status_code == 200:
            success = True
            break
        elif response.status_code == 403:  # Rate limit hit
            rotate_token()
            sleep(1)
        elif response.status_code == 404:
            break  # File not found, skip further attempts
        else:
            break  # Other error, skip

    if not success:
        continue

    df.at[idx, "has_manifest"] = "yes"

    try:
        content = response.json().get("content")
        if content:
            decoded = base64.b64decode(content).decode("utf-8")
            try:
                root = ET.fromstring(decoded)
                activities = root.findall(".//activity")
                if not activities:
                    # Handle namespaces if needed
                    ns_pattern = re.compile(r'\{(.+)\}')
                    match = ns_pattern.match(root.tag)
                    ns = {'android': match.group(1)} if match else {}
                    activities = root.findall(".//activity", namespaces=ns)
                if activities:
                    df.at[idx, "has_activity"] = "yes"
            except ET.ParseError:
                continue
    except Exception:
        continue

# Save output
df.to_csv(output_csv, index=False)
print(f"✅ Done. Results saved to: {output_csv}")


✅ Done. Results saved to: D:\Android_Mobile_App\AndroidProject_dataset\Repo_List_checked.csv
